# Exemplo fornecendo os artigos candidatos manualmente

In [35]:
import ollama
import pandas as pd
import spacy
import trafilatura

In [36]:
nlp = spacy.load("pt_core_news_lg")

In [37]:
df = pd.read_csv("candidatos.txt", header=None, names=["title", "url"])
df

,title,url
0,Ex-presidente do INSS Alessandro Stefanutto é ...,https://g1.globo.com/politica/noticia/2025/11/...
1,"Fraude no INSS: PF mirou em senador, dona de a...",https://g1.globo.com/politica/noticia/2025/12/...
2,Desvios no INSS: PF mira deputada e associação...,https://g1.globo.com/politica/noticia/2026/03/...
3,Entenda o esquema de fraudes no INSS que resul...,https://g1.globo.com/politica/noticia/2025/04/...
4,Fraude no INSS: esquema em associações envolve...,https://g1.globo.com/politica/noticia/2025/04/...
5,Fraude do INSS: beneficiários dizem que descon...,https://g1.globo.com/economia/noticia/2025/05/...
6,"Irã volta a fechar Estreito de Ormuz, atira em...",https://g1.globo.com/mundo/noticia/2026/04/18/...
7,Trump assina ordem para acelerar pesquisas e a...,https://g1.globo.com/saude/noticia/2026/04/18/...


In [38]:
df["content"] = df["url"].apply(trafilatura.fetch_url)
df["content"] = df["content"].apply(trafilatura.extract)
df["content"] = df["content"].fillna("")
df["content"]

0    O ex-presidente do Instituto Nacional do Segur...
1    A nova fase da Operação Sem Desconto, feita ne...
2    A Polícia Federal (PF) deflagrou, nesta terça-...
3    A Polícia Federal (PF) e a Controladoria-Geral...
4    Documentos da Polícia Federal (PF) e do Minist...
5    Sede do Instituto Nacional do Seguro Social, e...
6    O Irã subiu o tom neste sábado (18) em meio a ...
7    Trump durante evento na Casa Branca — Foto: Na...
Name: content, dtype: str

In [39]:
df["ents"] = df["content"].apply(lambda x: nlp(x).ents)
df["ents_labels"] = df["ents"].apply(lambda x: [ent.label_ for ent in x])
df["ents"] = df["ents"].apply(lambda ents: [ent.text for ent in ents])
df[["title", "ents", "ents_labels"]]

,title,ents,ents_labels
0,Ex-presidente do INSS Alessandro Stefanutto é ...,"[Instituto Nacional do Seguro Social, INSS, Al...","[LOC, ORG, PER, LOC, ORG, ORG, LOC, LOC, ORG, ..."
1,"Fraude no INSS: PF mirou em senador, dona de a...","[Polícia Federal, Antônio Carlos Camilo Antune...","[LOC, PER, PER, ORG, LOC, ORG, LOC, ORG, PER, ..."
2,Desvios no INSS: PF mira deputada e associação...,"[Polícia Federal, PF, Operação Indébito, INSS,...","[LOC, LOC, MISC, ORG, PER, LOC, ORG, ORG, PER,..."
3,Entenda o esquema de fraudes no INSS que resul...,"[Polícia Federal, PF, Controladoria-Geral da U...","[LOC, LOC, ORG, ORG, LOC, ORG, ORG, PER, ORG, ..."
4,Fraude no INSS: esquema em associações envolve...,"[Polícia Federal, PF, Ministério Público Feder...","[LOC, ORG, LOC, ORG, LOC, ORG, ORG, ORG, ORG, ..."
5,Fraude do INSS: beneficiários dizem que descon...,"[Instituto Nacional do Seguro Social, Brasília...","[LOC, LOC, MISC, ORG, LOC, ORG, LOC, ORG, ORG,..."
6,"Irã volta a fechar Estreito de Ormuz, atira em...","[Irã, EUA, Israel, Guarda Revolucionária, Estr...","[LOC, LOC, LOC, ORG, LOC, LOC, PER, LOC, PER, ..."
7,Trump assina ordem para acelerar pesquisas e a...,"[Trump, Casa Branca, Nathan Howard, Estados Un...","[PER, LOC, PER, LOC, PER, ORG, ORG, ORG, ORG, ..."


In [40]:
df.iloc[0]["ents_labels"][0]

'LOC'

In [41]:
def common_ents(ents_list):
    if not ents_list:
        return set()
    common = set(ents_list[0])
    for ents in ents_list[1:]:
        common.intersection_update(set(ents))
    return common
# teste
common_ents([["Alessandro Stefanutto", "INSS", "PF"], ["senador", "PF", "PT", "INSS"]])

{'INSS', 'PF'}

In [42]:
base_article = df.iloc[0]

common_rates = pd.DataFrame({
    "title": df["title"],
    "common_ents": df["ents"].apply(lambda ents: common_ents([base_article["ents"], ents]))
})
common_rates["common_count"] = common_rates["common_ents"].apply(len)
common_rates.sort_values("common_count", ascending=False, inplace=True)
common_rates

,title,common_ents,common_count
0,Ex-presidente do INSS Alessandro Stefanutto é ...,"{Senado, Confederação Brasileira dos Trabalhad...",87
3,Entenda o esquema de fraudes no INSS que resul...,"{INSS, PDT, Virgílio Antônio Ribeiro de Olivei...",30
5,Fraude do INSS: beneficiários dizem que descon...,"{Meu INSS, INSS, PDT, PF, Previdência, g1, Con...",14
1,"Fraude no INSS: PF mirou em senador, dona de a...","{Careca, INSS, Operação Sem Desconto, PF, Jeff...",12
2,Desvios no INSS: PF mira deputada e associação...,"{Jefferson Rudy, PF, INSS, Distrito Federal, g...",10
4,Fraude no INSS: esquema em associações envolve...,"{PF, INSS, Controladoria-Geral da União, CGU, ...",6
6,"Irã volta a fechar Estreito de Ormuz, atira em...",{Foto:},1
7,Trump assina ordem para acelerar pesquisas e a...,{},0


# Exemplo usando um motor de busca para obter os artigos candidatos

In [43]:
from ddgs import DDGS

In [44]:
query = "Caso INSS site:g1.globo.com"
ddgs = DDGS()
results = ddgs.text(query, max_results=20, region="br-pt")
results_df = pd.DataFrame(results)
results_df

,title,href,body
0,O Assunto #1454: INSS - a fraude bilionária da...,https://g1.globo.com/podcast/o-assunto/noticia...,"April 25, 2025 -Um dia depois de a Polícia Fed..."
1,Fraude no INSS: saiba quem são os suspeitos de...,https://g1.globo.com/politica/noticia/2025/04/...,Polícia Federal prendeu seis pessoas e outros ...
2,Fraude no INSS: veja lista de entidades suspei...,https://g1.globo.com/politica/noticia/2025/04/...,Uma operação realizada nesta quarta-feira (23)...
3,Fraudes no INSS: o que a PF descobriu no caso ...,https://g1.globo.com/politica/noticia/2025/05/...,Uma investigação da PF revelou desvios irregul...
4,Fraude do INSS: descontos ilegais começaram an...,https://g1.globo.com/economia/noticia/2025/05/...,Aposentados e pensionistas do Instituto Nacion...
5,"Fraude do INSS: 3,1 milhões já acusaram descon...",https://g1.globo.com/economia/noticia/2025/06/...,"Segundo Waller, o INSS já recebeu 3,1 milhões ..."
6,Governo prorroga até junho prazo de contestaçã...,https://g1.globo.com/economia/noticia/2026/03/...,"3 weeks ago -Segundo a PF,associações que ofer..."
7,Presidente do INSS Gilberto Waller é demitido;...,https://g1.globo.com/politica/noticia/2026/04/...,1 week ago -Gilberto Waller foi nomeado presid...
8,PF prende o 'Careca do INSS' em operação contr...,https://g1.globo.com/sp/sao-paulo/noticia/2025...,Uma investigação da PF revelou um amplo esquem...
9,Fraude no INSS: vítimas que aderirem ao ressar...,https://g1.globo.com/economia/noticia/2025/07/...,Aposentados e pensionistas que sofreram descon...


In [45]:
results_df["content"] = results_df["href"].apply(trafilatura.fetch_url)
results_df["content"] = results_df["content"].apply(trafilatura.extract)
results_df["content"] = results_df["content"].fillna("")
results_df["content"]

0     Um dia depois de a Polícia Federal e a Control...
1     Veja quem são os servidores afastados do INSS ...
2     Uma operação realizada nesta quarta-feira (23)...
3     O ministro da Previdência Social, Carlos Lupi,...
4     Sede do Instituto Nacional do Seguro Social, e...
5     O presidente do INSS, Gilberto Waller Júnior, ...
6     O governo federal prorrogou o prazo para que a...
7     O presidente do Instituto Nacional do Seguro S...
8     A Polícia Federal prendeu nesta sexta-feira (1...
9     Aposentados e pensionistas que sofreram descon...
10    O Supremo Tribunal Federal (STF) recebeu o pri...
11    Plenário do Conselho Nacional de Justiça — Fot...
12    O ex-presidente do INSS Alessandro Stefanutto ...
13    Após a deflagração da operação da Polícia Fede...
14    O relator da Comissão Parlamentar Mista de Inq...
15    Operação da Polícia Federal em vários estados ...
16    Fábio Luís Lula da Silva, o Lulinha, filho do ...
17    A revelação, em meados de abril, de um esq

In [71]:
results_df["date"] = results_df["href"].apply(lambda url: trafilatura.fetch_url(url)).apply(trafilatura.extract_metadata).apply(lambda meta: meta.date if meta else None)
results_df[["title", "date"]]

,title,date
0,O Assunto #1454: INSS - a fraude bilionária da...,2025-04-25
1,Fraude no INSS: saiba quem são os suspeitos de...,2025-04-24
2,Fraude no INSS: veja lista de entidades suspei...,2025-04-24
3,Fraudes no INSS: o que a PF descobriu no caso ...,2025-05-02
4,Fraude do INSS: descontos ilegais começaram an...,2025-05-06
5,"Fraude do INSS: 3,1 milhões já acusaram descon...",2025-06-13
6,Governo prorroga até junho prazo de contestaçã...,2026-03-27
7,Presidente do INSS Gilberto Waller é demitido;...,2026-04-13
8,PF prende o 'Careca do INSS' em operação contr...,2025-09-12
9,Fraude no INSS: vítimas que aderirem ao ressar...,2025-07-21


In [46]:
results_df["ents"] = results_df["content"].apply(lambda x: nlp(x).ents)
results_df["ents_labels"] = results_df["ents"].apply(lambda x: [ent.label_ for ent in x])
results_df["ents"] = results_df["ents"].apply(lambda ents: [ent.text for ent in ents])
results_df[["title", "ents", "ents_labels"]]

,title,ents,ents_labels
0,O Assunto #1454: INSS - a fraude bilionária da...,"[Polícia Federal, Controladoria-Geral da União...","[LOC, ORG, ORG, PER, PER, PER, PER, MISC, PER,..."
1,Fraude no INSS: saiba quem são os suspeitos de...,"[INSS, Justiça, Foto:, Instituto Nacional do S...","[ORG, ORG, MISC, LOC, ORG, LOC, ORG, ORG, ORG,..."
2,Fraude no INSS: veja lista de entidades suspei...,"[Polícia Federal, PF, Controladoria-Geral da U...","[LOC, ORG, ORG, ORG, LOC, ORG, ORG, LOC, ORG, ..."
3,Fraudes no INSS: o que a PF descobriu no caso ...,"[Previdência Social, Carlos Lupi, Polícia Fede...","[PER, PER, LOC, ORG, LOC, ORG, ORG, ORG, PER, ..."
4,Fraude do INSS: descontos ilegais começaram an...,"[Instituto Nacional do Seguro Social, Brasília...","[LOC, LOC, MISC, ORG, LOC, ORG, LOC, ORG, ORG,..."
5,"Fraude do INSS: 3,1 milhões já acusaram descon...","[INSS, Gilberto Waller Júnior, Correios, São P...","[ORG, PER, LOC, LOC, PER, ORG, LOC, ORG, LOC, ..."
6,Governo prorroga até junho prazo de contestaçã...,"[Instituto Nacional do Seguro Social, INSS, g1...","[LOC, ORG, ORG, MISC, ORG, ORG, ORG, ORG, MISC..."
7,Presidente do INSS Gilberto Waller é demitido;...,"[Instituto Nacional do Seguro Social, INSS, Gi...","[LOC, ORG, PER, PER, PER, MISC, PER, MISC, PER..."
8,PF prende o 'Careca do INSS' em operação contr...,"[Polícia Federal, Antônio Carlos Camilo Antune...","[LOC, PER, PER, PER, ORG, ORG, ORG, LOC, ORG, ..."
9,Fraude no INSS: vítimas que aderirem ao ressar...,"[Aposentados, Instituto Nacional do Seguro Soc...","[MISC, LOC, ORG, ORG, PER, LOC, MISC, MISC, MI..."


In [47]:
base_article = results_df.iloc[0]

common_rates = pd.DataFrame({
    "title": results_df["title"],
    "common_ents": results_df["ents"].apply(lambda ents: common_ents([base_article["ents"], ents]))
})
common_rates["common_count"] = common_rates["common_ents"].apply(len)
common_rates.sort_values("common_count", ascending=False, inplace=True)
common_rates

,title,common_ents,common_count
0,O Assunto #1454: INSS - a fraude bilionária da...,"{Diego Cherulli, INSS, Lula, Instituto Brasile...",28
17,"INSS: da lei de 1991 ao escândalo de 2025, rel...","{Luiz Felipe Silva, Mônica Mariotti, INSS, Lup...",14
12,A engrenagem de fraudes doINSS- O Assunto #159...,"{Diego Cherulli, Luiz Felipe Silva, INSS, Môni...",13
3,Fraudes no INSS: o que a PF descobriu no caso ...,"{INSS, Lupi, Lula, Controladoria-Geral da Uniã...",7
4,Fraude do INSS: descontos ilegais começaram an...,"{INSS, Lupi, Controladoria-Geral da União, Car...",7
1,Fraude no INSS: saiba quem são os suspeitos de...,"{INSS, Stefanutto, Controladoria-Geral da Uniã...",6
2,Fraude no INSS: veja lista de entidades suspei...,"{INSS, Stefanutto, Controladoria-Geral da Uniã...",6
15,O que a PF descobriu na investigação das fraud...,"{INSS, Lula, Controladoria-Geral da União, Car...",6
13,Fraude noINSSgera onda de desinformação nas re...,"{INSS, Lula, Controladoria-Geral da União, Car...",6
7,Presidente do INSS Gilberto Waller é demitido;...,"{INSS, Lula, presidente Lula, Alessandro Stefa...",5


In [51]:
results_df["content"]

0     Um dia depois de a Polícia Federal e a Control...
1     Veja quem são os servidores afastados do INSS ...
2     Uma operação realizada nesta quarta-feira (23)...
3     O ministro da Previdência Social, Carlos Lupi,...
4     Sede do Instituto Nacional do Seguro Social, e...
5     O presidente do INSS, Gilberto Waller Júnior, ...
6     O governo federal prorrogou o prazo para que a...
7     O presidente do Instituto Nacional do Seguro S...
8     A Polícia Federal prendeu nesta sexta-feira (1...
9     Aposentados e pensionistas que sofreram descon...
10    O Supremo Tribunal Federal (STF) recebeu o pri...
11    Plenário do Conselho Nacional de Justiça — Fot...
12    O ex-presidente do INSS Alessandro Stefanutto ...
13    Após a deflagração da operação da Polícia Fede...
14    O relator da Comissão Parlamentar Mista de Inq...
15    Operação da Polícia Federal em vários estados ...
16    Fábio Luís Lula da Silva, o Lulinha, filho do ...
17    A revelação, em meados de abril, de um esq

In [49]:
import numpy as np


def compute_similarity(text1, text2):
    client = ollama.Client("http://192.168.0.12:11434")
    embeddings = client.embed(
        model="nomic-embed-text-v2-moe",
        input=[text1, text2]
    )
    text1_vector = embeddings["embeddings"][0]
    text2_vector = embeddings["embeddings"][1]
    similarity = np.dot(text1_vector, text2_vector) / (np.linalg.norm(text1_vector) * np.linalg.norm(text2_vector))
    return similarity

# testando
print(compute_similarity("João foi ao mercado comprar frutas", "Maria foi ao mercado comprar frutas"))
print(compute_similarity("João foi ao mercado comprar frutas", "Pelé foi um grande jogador de futebol."))

0.7938537727910763
0.15262075528545455


In [53]:
def summarize_text(text):
    client = ollama.Client("http://192.168.0.12:11434")
    response = client.generate(
        model="deepseek-r1:8b",
        prompt=f"Resuma o seguinte texto:\n\n{text}\n\nResponda somente com o resumo.\nResumo:"
    )
    return response.response

In [54]:
results_df["summary"] = results_df["content"].apply(summarize_text)
results_df[["title", "summary"]]

,title,summary
0,O Assunto #1454: INSS - a fraude bilionária da...,Resumo:\n\nOperação da Polícia Federal e Contr...
1,Fraude no INSS: saiba quem são os suspeitos de...,**Resumo:**\n\nUma operação conjunta da Políci...
2,Fraude no INSS: veja lista de entidades suspei...,**Resumo:**\n\nA Polícia Federal (PF) e a Cont...
3,Fraudes no INSS: o que a PF descobriu no caso ...,## Resumo:\n\nO ministro da Previdência Social...
4,Fraude do INSS: descontos ilegais começaram an...,**Resumo:**\n\nO texto relata um amplo esquema...
5,"Fraude do INSS: 3,1 milhões já acusaram descon...","**Resumo:** \nO presidente do INSS, Gilberto ..."
6,Governo prorroga até junho prazo de contestaçã...,**Resumo:**\n\nO governo federal prorrogou o p...
7,Presidente do INSS Gilberto Waller é demitido;...,**Resumo:**\n\nA direção do Instituto Nacional...
8,PF prende o 'Careca do INSS' em operação contr...,**Resumo:**\n\nA Polícia Federal prendeu nesta...
9,Fraude no INSS: vítimas que aderirem ao ressar...,**Resumo:**\n\nAposentados e pensionistas que ...


In [58]:
base_article = results_df.iloc[0]
results_df["similarity"] = results_df["summary"].apply(lambda x: compute_similarity(base_article["summary"], x))
results_df[["title", "similarity"]].sort_values("similarity", ascending=False)

,title,similarity
0,O Assunto #1454: INSS - a fraude bilionária da...,1.000000
15,O que a PF descobriu na investigação das fraud...,0.821002
12,A engrenagem de fraudes doINSS- O Assunto #159...,0.786512
3,Fraudes no INSS: o que a PF descobriu no caso ...,0.762229
1,Fraude no INSS: saiba quem são os suspeitos de...,0.756236
2,Fraude no INSS: veja lista de entidades suspei...,0.742159
17,"INSS: da lei de 1991 ao escândalo de 2025, rel...",0.739621
4,Fraude do INSS: descontos ilegais começaram an...,0.727539
8,PF prende o 'Careca do INSS' em operação contr...,0.668444
18,Fraude no INSS: veja núcleos do grupo em que a...,0.630862


In [60]:
results_df["common_count"] = common_rates["common_count"]

In [62]:
results_df[["title", "similarity", "common_count"]].sort_values("similarity", ascending=False)

,title,similarity,common_count
0,O Assunto #1454: INSS - a fraude bilionária da...,1.000000,28
15,O que a PF descobriu na investigação das fraud...,0.821002,6
12,A engrenagem de fraudes doINSS- O Assunto #159...,0.786512,13
3,Fraudes no INSS: o que a PF descobriu no caso ...,0.762229,7
1,Fraude no INSS: saiba quem são os suspeitos de...,0.756236,6
2,Fraude no INSS: veja lista de entidades suspei...,0.742159,6
17,"INSS: da lei de 1991 ao escândalo de 2025, rel...",0.739621,14
4,Fraude do INSS: descontos ilegais começaram an...,0.727539,7
8,PF prende o 'Careca do INSS' em operação contr...,0.668444,5
18,Fraude no INSS: veja núcleos do grupo em que a...,0.630862,3


In [65]:
def pairwise_ranking(text1, text2):
    prompt = f"Compare os seguintes textos, o texto B tem relação de continuidade com o texto A? Responda com 'Sim' ou 'Não'.\n\nTexto A:\n{text1}\n\nTexto B:\n{text2}\n\nResposta:"
    client = ollama.Client("http://192.168.0.12:11434")
    response = client.generate(
        model="deepseek-r1:8b",
        prompt=prompt
    )
    return response.response.strip()

# testando
print(pairwise_ranking("O INSS anunciou um novo benefício para aposentados.", "O INSS lançou um novo programa de auxílio para aposentados."))
print(pairwise_ranking("O INSS anunciou um novo benefício para aposentados.", "O INSS enfrenta críticas por atrasos em pagamentos do novo benefício."))
print(pairwise_ranking("O INSS anunciou um novo benefício para aposentados.", "Feijão com farinha."))

Não
Sim
Não


In [66]:
results_df["continuity"] = results_df["summary"].apply(lambda x: pairwise_ranking(base_article["summary"], x))
results_df[["title", "similarity", "common_count", "continuity"]].sort_values("continuity", ascending=False)

,title,similarity,common_count,continuity
0,O Assunto #1454: INSS - a fraude bilionária da...,1.000000,28,Sim
1,Fraude no INSS: saiba quem são os suspeitos de...,0.756236,6,Sim
17,"INSS: da lei de 1991 ao escândalo de 2025, rel...",0.739621,14,Sim
16,"'Filho do rapaz', 'sócio oculto', nome em enve...",0.550632,4,Sim
15,O que a PF descobriu na investigação das fraud...,0.821002,6,Sim
14,CPMI do INSS: relator pede indiciamento de mai...,0.586485,5,Sim
13,Fraude noINSSgera onda de desinformação nas re...,0.615689,6,Sim
11,Fraude no INSS: CNJ analisará caso de juiz que...,0.435070,2,Sim
10,Fraude no INSS: STF recebe primeiro inquérito ...,0.559757,2,Sim
8,PF prende o 'Careca do INSS' em operação contr...,0.668444,5,Sim


In [72]:
results_df[["title", "date"]].sort_values("date")

,title,date
1,Fraude no INSS: saiba quem são os suspeitos de...,2025-04-24
2,Fraude no INSS: veja lista de entidades suspei...,2025-04-24
0,O Assunto #1454: INSS - a fraude bilionária da...,2025-04-25
15,O que a PF descobriu na investigação das fraud...,2025-04-29
3,Fraudes no INSS: o que a PF descobriu no caso ...,2025-05-02
4,Fraude do INSS: descontos ilegais começaram an...,2025-05-06
17,"INSS: da lei de 1991 ao escândalo de 2025, rel...",2025-05-06
13,Fraude noINSSgera onda de desinformação nas re...,2025-05-07
11,Fraude no INSS: CNJ analisará caso de juiz que...,2025-05-10
5,"Fraude do INSS: 3,1 milhões já acusaram descon...",2025-06-13


In [74]:
results_df[["title", "date", "similarity", "common_count", "continuity"]][results_df["continuity"] == "Sim"].sort_values("date")

,title,date,similarity,common_count,continuity
1,Fraude no INSS: saiba quem são os suspeitos de...,2025-04-24,0.756236,6,Sim
2,Fraude no INSS: veja lista de entidades suspei...,2025-04-24,0.742159,6,Sim
0,O Assunto #1454: INSS - a fraude bilionária da...,2025-04-25,1.000000,28,Sim
15,O que a PF descobriu na investigação das fraud...,2025-04-29,0.821002,6,Sim
4,Fraude do INSS: descontos ilegais começaram an...,2025-05-06,0.727539,7,Sim
17,"INSS: da lei de 1991 ao escândalo de 2025, rel...",2025-05-06,0.739621,14,Sim
13,Fraude noINSSgera onda de desinformação nas re...,2025-05-07,0.615689,6,Sim
11,Fraude no INSS: CNJ analisará caso de juiz que...,2025-05-10,0.435070,2,Sim
10,Fraude no INSS: STF recebe primeiro inquérito ...,2025-06-17,0.559757,2,Sim
8,PF prende o 'Careca do INSS' em operação contr...,2025-09-12,0.668444,5,Sim
